# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SULAIMAN-5-AHMED/FlyRankWeek1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import pandas as pd

HF_TOKEN = os.environ.get("HF_TOKEN")

from datasets import load_dataset
ds = load_dataset("FlyRank/internship-warehouse", "fact_content_daily_performance", streaming=True, split="train")
ds

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

IterableDataset({
    features: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'],
    num_shards: 18
})

In [20]:
ds= list(ds.take(10000))
ds = pd.DataFrame(ds)
print(ds.columns.tolist())
ds.head()

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115.0,...,0,0,0,0,0,0,0,0,0,0
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358.0,...,0,0,0,0,0,0,0,0,0,0
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34.0,...,0,0,0,0,0,0,0,0,0,0
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140.0,...,0,0,0,0,0,0,0,0,0,0
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89.0,...,0,0,0,0,0,0,0,0,0,0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*
gsc_impressions → visibility in search results

gsc_clicks → user interest

gsc_avg_position → ranking position

ga4_total_engagement_sec → engagement depth

sessions_organic, sessions_direct, sessions_referral, sessions_social, sessions_paid, sessions_ai → traffic source mix

# Why: These directly capture ranking signals and downstream engagement, which are predictive of site performance.

In [32]:
df_filtered = ds[['gsc_impressions','gsc_clicks', 'gsc_avg_position', 'ga4_total_engagement_sec','sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai' ]]
df_filtered.head(20)

,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_total_engagement_sec,sessions_organic,sessions_direct,sessions_referral,sessions_social,sessions_paid,sessions_ai
0,30,0,3.833333,0,0,0,0,0,0,0
1,5,0,71.600000,0,0,0,0,0,0,0
2,1,0,34.000000,0,0,0,0,0,0,0
3,6,0,23.333333,0,0,0,0,0,0,0
4,5,0,17.800000,0,0,0,0,0,0,0
5,21,0,50.000000,0,0,0,0,0,0,0
6,13,0,9.769231,0,0,0,0,0,0,0
7,29,0,12.275862,0,0,0,0,0,0,0
8,5,0,20.600000,0,0,0,0,0,0,0
9,8,0,38.000000,0,0,0,0,0,0,0


In [37]:
df_filtered["CTR"] = df_filtered["gsc_clicks"] / df_filtered["gsc_impressions"]
df_filtered = df_filtered.sort_values(by="CTR", ascending=False)
df_filtered.head(20)

/tmp/ipykernel_5270/674036709.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered["CTR"] = df_filtered["gsc_clicks"] / df_filtered["gsc_impressions"]


,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_total_engagement_sec,sessions_organic,sessions_direct,sessions_referral,sessions_social,sessions_paid,sessions_ai,CTR
9410,1,1,0.0,0,0,0,0,0,0,0,1.0
1460,1,1,2.0,0,0,0,0,0,0,0,1.0
8779,1,1,2.0,0,0,0,0,0,0,0,1.0
8699,1,1,1.0,0,0,0,0,0,0,0,1.0
6192,1,1,44.0,0,0,0,0,0,0,0,1.0
6221,2,2,4.5,0,0,0,0,0,0,0,1.0
6874,1,1,4.0,0,0,0,0,0,0,0,1.0
6871,1,1,57.0,0,0,0,0,0,0,0,1.0
8979,1,1,11.0,0,0,0,0,0,0,0,1.0
6658,1,1,0.0,0,0,0,0,0,0,0,1.0


In [39]:
(df_filtered[["ga4_total_engagement_sec",	'sessions_organic','sessions_direct','sessions_referral','sessions_social','sessions_paid','sessions_ai']] == 0).all()


,0
ga4_total_engagement_sec,True
sessions_organic,True
sessions_direct,True
sessions_referral,True
sessions_social,True
sessions_paid,True
sessions_ai,True


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

1) Grain is client_hash_id + content_hash_id + report_date  
2)  Dataset has ~10,000 rows in sample  
3) No missing values after fillna  
4) CTR is valid and bounded between 0–1  
5) sessions_ and ga4_total_engagement_sec are all zeros*

In [41]:
# claim 1

ds.groupby(["client_hash_id", "content_hash_id", "report_date"]).size().max()
# If the result is 1, then the grain is unique. If >1, you have duplicates.

1

In [42]:
# claim 2
len(df_filtered)


10000

In [43]:
#claim 3
df_filtered.isna().sum()

,0
gsc_impressions,0
gsc_clicks,0
gsc_avg_position,0
ga4_total_engagement_sec,0
sessions_organic,0
sessions_direct,0
sessions_referral,0
sessions_social,0
sessions_paid,0
sessions_ai,0


In [44]:
# claim 4
df_filtered["CTR"].describe()


,CTR
count,10000.000000
mean,0.010478
std,0.061232
min,0.000000
25%,0.000000
50%,0.000000
75%,0.000000
max,1.000000


In [36]:
df_filtered['gsc_avg_position'] = df_filtered['gsc_avg_position'].fillna(df_filtered['gsc_avg_position'].mean())
df_filtered.isna().sum()

/tmp/ipykernel_5270/154423340.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['gsc_avg_position'] = df_filtered['gsc_avg_position'].fillna(df_filtered['gsc_avg_position'].mean())


,0
gsc_impressions,0
gsc_clicks,0
gsc_avg_position,0
ga4_total_engagement_sec,0
sessions_organic,0
sessions_direct,0
sessions_referral,0
sessions_social,0
sessions_paid,0
sessions_ai,0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*
Unbalanced history

1) Early rows may only have Google Search Console (GSC) signals, with GA4/session fields all zeros.

In [46]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df_filtered[["ga4_total_engagement_sec","sessions_organic","sessions_direct",
             "sessions_referral","sessions_social","sessions_paid","sessions_ai"]].sum()



,0
ga4_total_engagement_sec,0
sessions_organic,0
sessions_direct,0
sessions_referral,0
sessions_social,0
sessions_paid,0
sessions_ai,0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.